### A5.4.1. Build Tensor Class from Scratch

**Problem:**

Build a class that represents a multi-dimensional array (tensor) backed by a flat list of numbers, with shape metadata and element access by multi-dimensional index.

**Explanation:**

A tensor stores numbers in a contiguous flat buffer. The shape tuple tells you how to interpret the flat buffer as a multi-dimensional grid. To access an element by multi-dimensional index, you compute a flat offset from the index and the strides.

How to build it:

1. Store `shape` (a tuple of dimension sizes) and `data` (a flat list of numbers) in `__init__`.
2. Compute `strides` from the shape. Strides tell you how many elements to skip per dimension. For row-major order, the last stride is 1, and each earlier stride is the product of all later dimension sizes.
3. Implement `__getitem__` that takes a tuple of indices, computes the flat offset as the dot product of indices and strides, and returns `data[offset]`.
4. Implement `__setitem__` similarly.
5. Provide a `size` property that returns the total number of elements.

**Example:**

A tensor with shape `(2, 3)` stores 6 elements. Element `[1, 2]` maps to flat index `1*3 + 2 = 5`.

In [1]:
from functools import reduce
import operator


class Tensor:
    def __init__(self, shape, data=None):
        self.shape = tuple(shape)
        self.strides = self._compute_strides(self.shape)
        total = reduce(operator.mul, self.shape, 1)
        self.data = list(data) if data else [0.0] * total

    def _compute_strides(self, shape):
        strides = [1] * len(shape)
        for dim in range(len(shape) - 2, -1, -1):
            strides[dim] = strides[dim + 1] * shape[dim + 1]
        return tuple(strides)

    def _flat_index(self, index):
        return sum(
            idx * stride
            for idx, stride in zip(index, self.strides)
        )

    def __getitem__(self, index):
        return self.data[self._flat_index(index)]

    def __setitem__(self, index, value):
        self.data[self._flat_index(index)] = value

    @property
    def size(self):
        return len(self.data)

    def __repr__(self):
        return f"Tensor(shape={self.shape}, data={self.data})"


tensor = Tensor((2, 3), [1, 2, 3, 4, 5, 6])
print(f"Shape: {tensor.shape}")
print(f"Strides: {tensor.strides}")
print(f"Element [0, 0]: {tensor[0, 0]}")
print(f"Element [1, 2]: {tensor[1, 2]}")

tensor[0, 1] = 99
print(f"After setting [0,1]=99: {tensor}")

tensor_3d = Tensor((2, 3, 4))
print(f"\n3D tensor strides: {tensor_3d.strides}")
print(f"3D tensor size: {tensor_3d.size}")

Shape: (2, 3)
Strides: (3, 1)
Element [0, 0]: 1
Element [1, 2]: 6
After setting [0,1]=99: Tensor(shape=(2, 3), data=[1, 99, 3, 4, 5, 6])

3D tensor strides: (12, 4, 1)
3D tensor size: 24


**References:**

📘 NumPy Documentation — [Internal Memory Layout of an ndarray](https://numpy.org/doc/stable/reference/arrays.ndarray.html#internal-memory-layout-of-an-ndarray)

---

[Next: Build Shape Inference from Scratch ➡️](./02_build_shape_inference_from_scratch.ipynb)